# HU-38 Calcular las correlaciones espaciales y consolidar la base analítica

### HU-38 consolidar la base analítica

In [ ]:
# libreria para procesamiento de archivos raster (TIFF)
%pip install numpy
%pip install pandas
%pip install geopandas
%pip install rasterio
%pip install --upgrade "pyarrow>=16.0.0"
%pip install "affine==2.4.0"

StatementMeta(, 97523a8f-59e0-4709-94a6-41fdf58db7a9, 13, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.1/368.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 111.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 126.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.3/33.3 MB 85.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 142.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 30.1 MB/s eta 0:00:00
  Attempting uninstall: python-da

In [ ]:
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# ETL HU-38 - CONSOLIDACIÓN DE ARCHIVOS TIFF EN MICROSOFT FABRIC
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# Objetivo:
# 1. Crear las carpetas de consolidación.
# 2. Consultar las cuatro fuentes.
# 3. Procesar únicamente archivos TIFF.
# 4. Omitir archivos cuyo nombre ya exista en el destino.
# 5. Validar y copiar archivos nuevos.
# 6. Registrar los resultados de cada archivo.
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

from notebookutils import fs
from rasterio.io import MemoryFile
from rasterio.windows import Window
from datetime import datetime, timezone
import json


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 1. CONFIGURACIÓN DE RUTAS Y FUENTES
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

BASE = "Files"
DESTINO = f"{BASE}/CONSOLIDADO"

FUENTES = {
    "TCH_TCHM": "Data_MapasProductividad",
    "SUELOS_QUIMICOS": "Geodata_Fertilizacion",
    "COSECHA_HISTORICA": "RASTER_HISTORICOS",
    "DESPOBLACION": "Rosemberg_despoblacion"
}

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 1/7] Configuración de rutas y fuentes completada.\n")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 2. CREACIÓN DE CARPETAS
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

ejecucion = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
TEMPORAL = f"{DESTINO}/_TEMP_ETL/{ejecucion}"
resultados = []

# Crear la carpeta raíz de consolidación.
fs.mkdirs(DESTINO)

# Crear una carpeta independiente para cada fuente.
for nombre_fuente in FUENTES:
    fs.mkdirs(f"{DESTINO}/{nombre_fuente}")

# Crear la carpeta temporal de esta ejecución.
fs.mkdirs(TEMPORAL)

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 2/7] Carpetas de consolidación y temporales preparadas.\n")
print(f"             Destino: {DESTINO}")
print(f"             Ejecución: {ejecucion}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 3. FUNCIÓN DE VALIDACIÓN TIFF
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

def validar_tiff(ruta):
    """
    Comprueba disponibilidad, tamaño, apertura raster y lectura
    de una pequeña ventana de píxeles.
    Devuelve el tamaño en bytes si supera las comprobaciones.
    """

    # Leer el contenido binario del archivo.
    registro = (
        spark.read.format("binaryFile")
        .load(ruta)
        .select("length", "content")
        .first()
    )

    # Verificar que el archivo esté disponible.
    if registro is None or registro["content"] is None:
        raise ValueError("Archivo no disponible o sin contenido")

    contenido = bytes(registro["content"])
    tamano = int(registro["length"])

    # Comprobar que el archivo no esté vacío y que su tamaño coincida.
    if tamano <= 0 or len(contenido) != tamano:
        raise ValueError("Archivo vacío o tamaño inconsistente")

    # Abrir el TIFF en memoria sin modificar el archivo original.
    with MemoryFile(contenido) as memoria:
        with memoria.open() as raster:

            # Verificar dimensiones y número de bandas.
            if (
                raster.width <= 0
                or raster.height <= 0
                or raster.count < 1
            ):
                raise ValueError("Estructura raster inválida")

            # Leer una ventana pequeña para comprobar la decodificación.
            ventana = Window(
                0,
                0,
                min(10, raster.width),
                min(10, raster.height)
            )

            pixeles = raster.read(1, window=ventana)

            if pixeles.size == 0:
                raise ValueError("No se pudieron leer píxeles")

    return tamano


print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 3/7] Función de validación TIFF preparada.\n")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 4. CONSULTA DE LAS FUENTES
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

for nombre_fuente, carpeta_origen in FUENTES.items():

    origen = f"{BASE}/{carpeta_origen}"
    destino = f"{DESTINO}/{nombre_fuente}"

    print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 4/7] Consultando fuente: {nombre_fuente}\n")

    try:
        archivos = fs.ls(origen)

        print(f"             Elementos encontrados: {len(archivos)}")

    except Exception as e:
        resultados.append({
            "fuente": nombre_fuente,
            "archivo": None,
            "estado": "ERROR_ORIGEN",
            "detalle": str(e)[:300]
        })

        print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[ERROR] No se pudo consultar la fuente {nombre_fuente}: {e}\n")
        print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 4/7] Fuente finalizada con error: {nombre_fuente}\n")
        continue

    # Contadores individuales para mostrar el resultado de la fuente.
    copiados_fuente = 0
    omitidos_fuente = 0
    errores_fuente = 0
    tiffs_fuente = 0

    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
    # FASE 5. VALIDACIÓN, COPIA Y PUBLICACIÓN
    #+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

    for archivo in archivos:

        nombre = archivo.name

        # Ignorar archivos que no sean TIFF.
        if not nombre.lower().endswith((".tif", ".tiff")):
            continue

        tiffs_fuente += 1

        ruta_origen = f"{origen}/{nombre}"
        ruta_destino = f"{destino}/{nombre}"
        ruta_temporal = f"{TEMPORAL}/{nombre_fuente}__{nombre}"

        try:

            # Omitir archivos cuyo nombre ya exista en el destino.
            if fs.exists(ruta_destino):

                resultados.append({
                    "fuente": nombre_fuente,
                    "archivo": nombre,
                    "estado": "OMITIDO_EXISTENTE"
                })

                omitidos_fuente += 1
                print(f"  [OMITIDO] {nombre} | Ya existe")
                continue

            # Validar el archivo original antes de copiarlo.
            tamano_origen = validar_tiff(ruta_origen)
            print(f"  [VALIDADO ORIGEN] {nombre}")

            # Copiar el archivo a la carpeta temporal de la ejecución.
            copia_ok = fs.cp(ruta_origen, ruta_temporal)

            if not copia_ok or not fs.exists(ruta_temporal):
                raise IOError("No se completó la copia temporal")

            print(f"  [COPIA TEMPORAL] {nombre}")

            # Validar el archivo temporal.
            tamano_temporal = validar_tiff(ruta_temporal)

            if tamano_temporal != tamano_origen:
                raise IOError("El tamaño de la copia no coincide")

            print(f"  [VALIDACIÓN TEMPORAL] {nombre}")

            # Volver a comprobar el destino para evitar sobrescribirlo.
            if fs.exists(ruta_destino):

                resultados.append({
                    "fuente": nombre_fuente,
                    "archivo": nombre,
                    "estado": "OMITIDO_EXISTENTE"
                })

                omitidos_fuente += 1
                print(f"  [OMITIDO] {nombre} | Destino ya creado")
                continue

            # Publicar la copia temporal sin permitir sobrescritura.
            publicado = fs.mv(
                ruta_temporal,
                ruta_destino,
                create_path=True,
                overwrite=False
            )

            if not publicado or not fs.exists(ruta_destino):
                raise IOError("No se confirmó la publicación")

            print(f"  [PUBLICACIÓN] {nombre}")

            # Validar el archivo ya publicado.
            tamano_final = validar_tiff(ruta_destino)

            if tamano_final != tamano_origen:
                raise IOError("El tamaño final no coincide")

            # Registrar la publicación exitosa.
            resultados.append({
                "fuente": nombre_fuente,
                "archivo": nombre,
                "estado": "COPIADO_VALIDADO",
                "bytes": tamano_final
            })

            copiados_fuente += 1
            print(f"  [OK] {nombre} | Copiado y validado \n+-+-+-+-+-")

        except Exception as e:

            # Registrar el fallo de este archivo y continuar con los demás.
            resultados.append({
                "fuente": nombre_fuente,
                "archivo": nombre,
                "estado": "ERROR",
                "detalle": str(e)[:300]
            })

            errores_fuente += 1
            print(f"  [ERROR] {nombre} | {e}")

    # Mensaje de cierre de la fuente actual.
    print(f"\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 5/7] Fuente finalizada: \n{nombre_fuente}")
    print(f"             TIFF encontrados: {tiffs_fuente}")
    print(f"             Copiados: {copiados_fuente}")
    print(f"             Omitidos: {omitidos_fuente}")
    print(f"             Errores: {errores_fuente}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 6. REGISTRO DE CONTROL
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

ruta_control = f"{DESTINO}/CONTROL_ETL_{ejecucion}.json"

fs.put(
    ruta_control,
    json.dumps(resultados, ensure_ascii=False, indent=2),
    overwrite=False
)

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 6/7] Registro de control guardado correctamente.\n")
print(f"             Archivo: {ruta_control}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 7. RESUMEN FINAL
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

copiados = sum(
    r["estado"] == "COPIADO_VALIDADO"
    for r in resultados
)

omitidos = sum(
    r["estado"] == "OMITIDO_EXISTENTE"
    for r in resultados
)

errores = sum(
    r["estado"].startswith("ERROR")
    for r in resultados
)

errores_origen = sum(
    r["estado"] == "ERROR_ORIGEN"
    for r in resultados
)

print("\n")
print("#" * 65)
print("ETL HU-38 | RESUMEN FINAL DE EJECUCIÓN")
print("#" * 65)
print(f"Archivos evaluados:       {len(resultados)}")
print(f"Copiados y validados:     {copiados}")
print(f"Omitidos por existencia:  {omitidos}")
print(f"Errores de archivos:      {errores}")
print(f"Errores de fuentes:       {errores_origen}")
print(f"Registro de control:      {ruta_control}")

if errores == 0 and errores_origen == 0:
    print("\nESTADO GENERAL: EJECUCIÓN COMPLETADA SIN ERRORES")
else:
    print("\nESTADO GENERAL: EJECUCIÓN COMPLETADA CON INCIDENCIAS")

print("\n#+-+--+--+-+-+--+--+-+-+--+--+-\n[FASE 7/7] Proceso ETL HU-38 finalizado.\n")



StatementMeta(, 97523a8f-59e0-4709-94a6-41fdf58db7a9, 15, Finished, Available, Finished, False)


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 1/7] Configuración de rutas y fuentes completada.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 2/7] Carpetas de consolidación y temporales preparadas.

             Destino: Files/CONSOLIDADO
             Ejecución: 20261030T161410Z

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 3/7] Función de validación TIFF preparada.


#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: TCH_TCHM

             Elementos encontrados: 2
  [OMITIDO] MP_20261021_010721012.tif | Ya existe
  [OMITIDO] TCHM_2026.tif | Ya existe

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 5/7] Fuente finalizada: 
TCH_TCHM
             TIFF encontrados: 2
             Copiados: 0
             Omitidos: 2
             Errores: 0

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 4/7] Consultando fuente: SUELOS_QUIMICOS

             Elementos encontrados: 1
  [OMITIDO] P_TCHM.tif | Ya existe

#+-+--+--+-+-+--+--+-+-+--+--+-
[FASE 5/7] Fuente finalizada: 
SUELOS_QUIMICOS
             TIFF encontrados: 1
   

### HU-38 Calcular las correlaciones espaciales

In [ ]:

#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# ETL HU-38 - CORRELACIONES ESPACIALES CONTRA TCHM DEL AÑO ACTUAL
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

from pathlib import Path
from datetime import datetime
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.mask import mask
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 1. CONFIGURACIÓN
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

ANIO_ACTUAL = datetime.now().year

BASE = Path("/lakehouse/default/Files")
RUTA_SHAPEFILE = BASE / "capas" / "ste.shp"

CARPETA_CONSOLIDADO = BASE / "CONSOLIDADO"
CARPETA_TCHM = CARPETA_CONSOLIDADO / "TCH_TCHM"

CARPETA_SALIDA = (
    CARPETA_CONSOLIDADO / "CORRELACIONES_ESPACIALES"
)
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

print(f"[FASE 1/7] Año de referencia: {ANIO_ACTUAL}")
print(f"             Shapefile: {RUTA_SHAPEFILE}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 2. FUNCIONES DE IDENTIFICACIÓN
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(
        c for c in texto if not unicodedata.combining(c)
    )
    return re.sub(r"[^a-z0-9]", "", texto.lower())


# Alias ordenados desde los nombres más específicos.
# Esto evita confundir lluvias de periodos distintos.
ALIAS_VARIABLES = {
    "Lluvias (0 - 3 Meses)": [
        "LLUVIAS_0_-_3_MESES",
        "LLUVIAS_0_3_MESES",
        "LLUVIAS03MESES"
    ],
    "Lluvias 10d Ant.Cos": [
        "LLUVIAS_10D_ANTCOS",
        "LLUVIAS10DANTCOS"
    ],
    "Variedad_TIFF": ["VARIEDAD_TIFF"],
    "Semanas mad.": ["SEMANAS_MAD"],
    "Num.Riegos": ["NUMRIEGOS", "NUM_RIEGOS"],
    "Ton.Azucar": ["TONAZUCAR", "TON_AZUCAR"],
    "%Sac.Cana": ["SACCANA", "SAC_CANA"],
    "Lluvias": ["LLUVIAS"],
    "Corte": ["CORTE"],
    "Edad": ["EDAD"],
    "Rdto": ["RDTO", "RENDIMIENTO"],
    "TAHM": ["TAHM"],
    "TAH": ["TAH"],
    "Vejez": ["VEJEZ"]
}


def identificar_variable(nombre_archivo):
    """
    Identifica la variable usando el nombre del archivo,
    sin confundir años ni periodos de lluvia.
    """
    nombre = Path(nombre_archivo).stem

    # Eliminar el año inicial, por ejemplo 2017_.
    nombre_sin_anio = re.sub(
        r"^\d{4}[_-]+", "", nombre
    )
    nombre_norm = normalizar(nombre_sin_anio)

    for variable, alias_lista in ALIAS_VARIABLES.items():
        for alias in alias_lista:
            if normalizar(alias) == nombre_norm:
                return variable

    return "OTRA"


def obtener_anio(nombre_archivo):
    coincidencia = re.match(
        r"^(\d{4})[_-]", Path(nombre_archivo).stem
    )
    return int(coincidencia.group(1)) if coincidencia else None


def nombre_columna(ruta):
    """
    Mantiene año y variable en el nombre para diferenciar
    los TIFF de distintos años.
    """
    return re.sub(
        r"[^A-Za-z0-9]+", "_", Path(ruta).stem
    ).strip("_")


def calcular_pearson(a, b):
    """
    Calcula Pearson con pares de píxeles válidos.
    """
    a = np.asarray(a, dtype="float64").ravel()
    b = np.asarray(b, dtype="float64").ravel()

    validos = np.isfinite(a) & np.isfinite(b)
    a = a[validos]
    b = b[validos]

    n = len(a)

    if n < 2:
        return np.nan, n

    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan, n

    return float(np.corrcoef(a, b)[0, 1]), n


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 3. VALIDAR ARCHIVOS Y CARGAR POLÍGONOS
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

if not RUTA_SHAPEFILE.exists():
    raise FileNotFoundError(
        f"No se encontró el shapefile: {RUTA_SHAPEFILE}"
    )

gdf_lotes = gpd.read_file(RUTA_SHAPEFILE)

if gdf_lotes.empty:
    raise ValueError("El shapefile no contiene polígonos.")

if gdf_lotes.crs is None:
    raise ValueError(
        "El shapefile no tiene sistema de coordenadas definido."
    )

campos = {str(c).upper(): c for c in gdf_lotes.columns}

if "CODSIA" not in campos:
    raise ValueError(
        f"No se encuentra CODSIA. Campos: {list(gdf_lotes.columns)}"
    )

CAMPO_LOTE = campos["CODSIA"]

# Buscar el raster de referencia del año actual.
candidatos_tchm = sorted(
    p for p in CARPETA_TCHM.iterdir()
    if p.is_file()
    and p.suffix.lower() in (".tif", ".tiff")
    and normalizar(p.stem) == normalizar(f"TCHM_{ANIO_ACTUAL}")
)

if len(candidatos_tchm) != 1:
    raise ValueError(
        f"Se esperaba exactamente un TCHM_{ANIO_ACTUAL}.tif "
        f"en {CARPETA_TCHM}; encontrados: "
        f"{[p.name for p in candidatos_tchm]}"
    )

RUTA_TCHM = candidatos_tchm[0]

print(f"[FASE 3/7] Lotes cargados: {len(gdf_lotes)}")
print(f"             Referencia: {RUTA_TCHM.name}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 4. IDENTIFICAR TODOS LOS TIFF CONSOLIDADOS
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

archivos_tiff = sorted(
    p for p in CARPETA_CONSOLIDADO.rglob("*")
    if p.is_file()
    and p.suffix.lower() in (".tif", ".tiff")
    and "_TEMP_ETL" not in p.parts
    and "CORRELACIONES_ESPACIALES" not in p.parts
)

# Excluir el raster TCHM de referencia de la lista comparativa.
archivos_comparar = [
    p for p in archivos_tiff
    if p.resolve() != RUTA_TCHM.resolve()
]

# Evitar que dos archivos con el mismo nombre de columna
# sobrescriban resultados en la matriz final.
columnas = [nombre_columna(p) for p in archivos_comparar]

if len(columnas) != len(set(columnas)):
    raise ValueError(
        "Hay TIFF con nombres repetidos en distintas carpetas. "
        "Se deben diferenciar por carpeta antes de generar la matriz."
    )

print(f"[FASE 4/7] TIFF encontrados: {len(archivos_tiff)}")
print(f"             TIFF por comparar: {len(archivos_comparar)}")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 5. PREPARAR LOS POLÍGONOS Y EL TCHM
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

resultados = []

with rasterio.open(RUTA_TCHM) as raster_ref:

    if raster_ref.crs is None:
        raise ValueError("El TCHM no tiene CRS definido.")

    # Proyectar los lotes al CRS de referencia.
    lotes_ref = gdf_lotes.to_crs(raster_ref.crs)

    # Preparar cada geometría y recortar el TCHM una sola vez.
    lotes_preparados = []

    for _, lote in lotes_ref.iterrows():
        geometria = lote.geometry
        codsia = lote[CAMPO_LOTE]

        if geometria is None or geometria.is_empty:
            lotes_preparados.append(
                (codsia, None, None)
            )
            continue

        try:
            ref_recortado, _ = mask(
                raster_ref,
                [geometria],
                crop=True,
                filled=False
            )
            lotes_preparados.append(
                (codsia, geometria, ref_recortado[0])
            )
        except ValueError:
            # El lote no intersecta el TCHM.
            lotes_preparados.append(
                (codsia, geometria, None)
            )

print("[FASE 5/7] Polígonos y ventanas del TCHM preparados.")


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 6. CORRELACIÓN DE CADA TIFF POR LOTE
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

for indice, ruta_variable in enumerate(archivos_comparar, start=1):

    variable = identificar_variable(ruta_variable.name)
    anio_raster = obtener_anio(ruta_variable.name)
    columna = nombre_columna(ruta_variable)

    print(
        f"[FASE 6/7] TIFF {indice}/{len(archivos_comparar)}: "
        f"{ruta_variable.name}"
    )

    try:
        with rasterio.open(ruta_variable) as raster_var:

            if raster_var.crs is None:
                raise ValueError("El TIFF no tiene CRS definido.")

            # Alinear el TIFF con la cuadrícula exacta del TCHM.
            with WarpedVRT(
                raster_var,
                crs=raster_ref.crs,
                transform=raster_ref.transform,
                width=raster_ref.width,
                height=raster_ref.height,
                resampling=Resampling.nearest
            ) as raster_alineado:

                for codsia, geometria, ref_recortado in lotes_preparados:

                    registro = {
                        "CODSIA": codsia,
                        "anio_referencia": ANIO_ACTUAL,
                        "raster": ruta_variable.name,
                        "anio_raster": anio_raster,
                        "variable": variable,
                        "correlacion_pearson": np.nan,
                        "n_pixeles": 0
                    }

                    if geometria is None or ref_recortado is None:
                        resultados.append(registro)
                        continue

                    try:
                        var_recortado, _ = mask(
                            raster_alineado,
                            [geometria],
                            crop=True,
                            filled=False
                        )

                        var_valores = var_recortado[0]

                        if ref_recortado.shape != var_valores.shape:
                            raise ValueError(
                                "Las ventanas no tienen igual dimensión."
                            )

                        # Excluir NoData de ambos rasters.
                        mascara_valida = (
                            ~np.ma.getmaskarray(ref_recortado)
                            & ~np.ma.getmaskarray(var_valores)
                        )

                        valores_ref = np.asarray(
                            ref_recortado.data
                        )[mascara_valida]

                        valores_var = np.asarray(
                            var_valores.data
                        )[mascara_valida]

                        correlacion, n_pixeles = calcular_pearson(
                            valores_ref,
                            valores_var
                        )

                        registro["correlacion_pearson"] = correlacion
                        registro["n_pixeles"] = n_pixeles

                    except ValueError as e:
                        warnings.warn(
                            f"Lote {codsia}, TIFF {ruta_variable.name}: {e}"
                        )

                    resultados.append(registro)

    except Exception as e:
        warnings.warn(
            f"No se pudo procesar {ruta_variable.name}: {e}"
        )

        # Registrar el fallo para cada lote y continuar con los demás TIFF.
        for codsia, _, _ in lotes_preparados:
            resultados.append({
                "CODSIA": codsia,
                "anio_referencia": ANIO_ACTUAL,
                "raster": ruta_variable.name,
                "anio_raster": anio_raster,
                "variable": variable,
                "correlacion_pearson": np.nan,
                "n_pixeles": 0,
                "error": str(e)[:300]
            })


#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
# FASE 7. EXPORTAR DETALLE Y MATRIZ
#+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

df_detalle = pd.DataFrame(resultados)

ruta_detalle = (
    CARPETA_SALIDA
    / f"CORRELACIONES_TCHM_{ANIO_ACTUAL}_DETALLE.csv"
)

df_detalle.to_csv(
    ruta_detalle,
    index=False,
    encoding="utf-8-sig"
)

# Crear matriz: una fila por lote y una columna por TIFF.
if not df_detalle.empty:
    df_matriz = df_detalle.pivot_table(
        index="CODSIA",
        columns="raster",
        values="correlacion_pearson",
        aggfunc="first"
    ).reset_index()

    df_matriz.columns.name = None
else:
    df_matriz = pd.DataFrame(columns=["CODSIA"])

ruta_matriz = (
    CARPETA_SALIDA
    / f"CORRELACIONES_TCHM_{ANIO_ACTUAL}_MATRIZ.csv"
)

df_matriz.to_csv(
    ruta_matriz,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "#+-" * 22)
print("HU-38 | RESUMEN DE CORRELACIONES ESPACIALES")
print("#+-" * 22)
print(f"Lotes: {len(gdf_lotes)}")
print(f"TIFF comparativos: {len(archivos_comparar)}")
print(f"Registros detalle: {len(df_detalle)}")
print(f"Archivo detalle: {ruta_detalle}")
print(f"Archivo matriz: {ruta_matriz}")

display(df_detalle.head(20))
display(df_matriz.head(20))


StatementMeta(, 97523a8f-59e0-4709-94a6-41fdf58db7a9, 16, Finished, Available, Finished, False)

[FASE 1/7] Año de referencia: 2026
             Shapefile: /lakehouse/default/Files/capas/ste.shp
[FASE 3/7] Lotes cargados: 9773
             Referencia: TCHM_2026.tif
[FASE 4/7] TIFF encontrados: 181
             TIFF por comparar: 180
[FASE 5/7] Polígonos y ventanas del TCHM preparados.
[FASE 6/7] TIFF 1/180: 2014_CORTE.tif
[FASE 6/7] TIFF 2/180: 2014_EDAD.tif
[FASE 6/7] TIFF 3/180: 2014_LLUVIAS_0_-_3_MESES.tif
[FASE 6/7] TIFF 4/180: 2014_LLUVIAS_10D_ANTCOS.tif
[FASE 6/7] TIFF 5/180: 2014_RDTO.tif
[FASE 6/7] TIFF 6/180: 2014_SACCANA.tif
[FASE 6/7] TIFF 7/180: 2014_SEMANAS_MAD.tif
[FASE 6/7] TIFF 8/180: 2014_TAH.tif
[FASE 6/7] TIFF 9/180: 2014_TAHM.tif
[FASE 6/7] TIFF 10/180: 2014_TONAZUCAR.tif
[FASE 6/7] TIFF 11/180: 2014_VARIEDAD_TIFF.tif
[FASE 6/7] TIFF 12/180: 2014_VEJEZ.tif
[FASE 6/7] TIFF 13/180: 2015_CORTE.tif
[FASE 6/7] TIFF 14/180: 2015_EDAD.tif
[FASE 6/7] TIFF 15/180: 2015_LLUVIAS_0_-_3_MESES.tif
[FASE 6/7] TIFF 16/180: 2015_LLUVIAS_10D_ANTCOS.tif
[FASE 6/7] TIFF 17/180: 20

SynapseWidget(Synapse.DataFrame, 12d6e6f4-cfe6-44f0-8201-8bf2f367ae1c)

SynapseWidget(Synapse.DataFrame, 1b70d692-63b2-43f9-99b1-00df8fc49137)

1. Resumen del archivo
Lotes analizados
20

Columnas de ráster
180

Celdas NULL
1.331
36,97 % de los valores

Valores negativos
1.144
31,78 % de los valores



2. ¿Por qué aparecen valores negativos?
Si el CSV contiene los coeficientes de correlación de Pearson entre cada ráster y TCHM_2026.tif, los valores negativos son válidos.
Valor	Interpretación
Cerca de +1	Relación positiva: ambas variables tienden a aumentar juntas.
Cerca de 0	No se observa una relación lineal clara.
Cerca de -1	Relación negativa: cuando una variable aumenta, la otra tiende a disminuir.
NULL	No fue posible calcular la correlación con los datos disponibles.

Por ejemplo, una correlación de -0,60 no significa que el ráster tenga valores negativos. Significa que existe una relación inversa entre sus valores y los del TCHM dentro del lote.

Importante: si el ráster contiene categorías, como VARIEDAD_TIFF o CORTE, una correlación de Pearson puede no ser la medida más adecuada, especialmente cuando los códigos representan categorías y no cantidades continuas.


3. ¿Por qué aparecen tantos NULL?
Las causas más probables en tu procesamiento son:
- Sin píxeles válidos coincidentes: el lote no tiene datos útiles en uno de los dos rásteres.
- Valores NoData: el ráster contiene píxeles sin información válida dentro del polígono.
- Valores constantes: si todos los píxeles válidos de un ráster tienen el mismo valor dentro de un lote, la desviación estándar es cero y Pearson no se puede calcular.
- Muy pocos píxeles válidos: aunque matemáticamente pueda calcularse una correlación con dos pares, el resultado no sería estadísticamente confiable.
